# 15 — Execution Module Quickstart

Execution guidance for screener output and order-type suggestions. See the [execution README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/execution/README.md) for full documentation.

In [ ]:
from __future__ import annotations

from swing_screener.execution.guidance import add_execution_guidance, ExecutionConfig, apply_pattern_stop
from swing_screener.indicators.candles import CandlePattern

import pandas as pd
pd.set_option("display.width", 140)

## ExecutionConfig

Controls how the guidance engine suggests order types and prices. Each field has a config default loaded from `config/defaults.yaml`.

In [ ]:
cfg = ExecutionConfig(
    breakout_stop_buffer_pct=0.002,
    pullback_atr_fraction=0.25,
    allow_second_chance_breakout=True,
    pattern_stop_enabled=True,
)
print(cfg)

## Execution Guidance on Screener Output

Generate suggested order types (limit/market), price bands, and execution notes from a screener-style DataFrame.

In [ ]:
data = {
    "ticker": ["AAPL", "NVDA", "MSFT"],
    "signal": ["breakout", "breakout", "pullback"],
    "last": [178.50, 485.00, 375.00],
    "breakout_level": [177.00, 480.00, 370.00],
    "ma20_level": [175.00, 470.00, 365.00],
    "atr14": [3.5, 12.0, 5.0],
}
df = pd.DataFrame(data)
df

In [ ]:
cfg = ExecutionConfig(breakout_stop_buffer_pct=0.002, pullback_atr_fraction=0.25)
result = add_execution_guidance(df, cfg)
result[[
    "ticker", "signal",
    "suggested_order_type", "suggested_order_price",
    "order_price_band_low", "order_price_band_high",
    "execution_note",
]]

## Pattern Stop

`apply_pattern_stop` tightens the stop when the latest bar shows a bullish
candlestick pattern (hammer, bullish_engulfing, inside_bar) at a breakout
or pullback context.

In [ ]:
pattern = CandlePattern(
    bar_index=0,
    date="2024-01-16",
    ticker="AAPL",
    name="hammer",
    direction="bullish",
    key_level=172.50,
    context="at_breakout",
)

new_stop, reason = apply_pattern_stop(
    ticker="AAPL",
    entry=175.50,
    current_stop=170.00,
    atr=3.5,
    patterns={"AAPL": [pattern]},
    buffer_atr=0.25,
    min_rr_stop=None,
)
print(f"Stop: {new_stop}, Reason: {reason}")